# TopologicPy Semantic Reasoning with `Reasoner.py`

This notebook demonstrates a lightweight reasoning layer for TopologicPy/TGraph. It shows how to:

1. Build a small semantic `TGraph`.
2. Export it to RDF.
3. Inject the TopologicPy ontology axioms.
4. Run deterministic RDFS reasoning.
5. Inspect inferred triples.
6. Write inferred classes back into TGraph dictionaries.

The default implementation only requires `rdflib`. If `owlrl` or `pyshacl` are installed, `Reasoner.Infer(profile="owlrl")` and `Reasoner.Validate(...)` can use them.

In [1]:
# Optional dependencies
# !pip install rdflib owlrl pyshacl

import sys
from pathlib import Path

# This cell is not needed if you have pip installed topologicpy
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

# If testing from a local folder containing TGraph.py, Ontology.py, and Reasoner.py:
# sys.path.insert(0, str(Path.cwd()))

try:
    from topologicpy.TGraph import TGraph
    from topologicpy.Ontology import Ontology
    from topologicpy.Reasoner import Reasoner
except Exception:
    # Fallback for testing this notebook beside the files.
    sys.path.insert(0, str(Path.cwd()))
    from TGraph import TGraph
    from Ontology import Ontology
    from Reasoner import Reasoner

Reasoner.Dependencies()

{'rdflib': {'available': True, 'version': '6.3.2'},
 'owlrl': {'available': False, 'error': "No module named 'owlrl'"},
 'pyshacl': {'available': False, 'error': "No module named 'pyshacl'"}}

## 1. Build a small semantic TGraph

The graph below contains two rooms and one door. The rooms and door are asserted only with their most specific classes. The reasoner will infer their superclasses, such as `top:Space`, `top:Zone`, `top:Element`, and BOT bridge classes such as `bot:Space` or `bot:Element`.

In [2]:
graph = TGraph(dictionary={"label": "Reasoning Demo Graph", "uri": "inst:demo_graph"})
TGraph.SetOntologyClass(graph, "top:KnowledgeGraph")

room_a = graph.AddVertex(dictionary={
    "label": "Room A",
    "uri": "inst:room_a",
    "ontology_class": "top:Room",
    "x": 0.0, "y": 0.0, "z": 0.0,
})

room_b = graph.AddVertex(dictionary={
    "label": "Room B",
    "uri": "inst:room_b",
    "ontology_class": "top:Room",
    "x": 5.0, "y": 0.0, "z": 0.0,
})

door = graph.AddVertex(dictionary={
    "label": "Door 1",
    "uri": "inst:door_1",
    "ontology_class": "top:Door",
    "x": 2.5, "y": 0.0, "z": 0.0,
})

graph.AddEdge(room_a, door, dictionary={
    "label": "Room A has door",
    "uri": "inst:edge_room_a_door",
    "ontology_class": "top:Relationship",
    "relationship": "hasDoor",
})

graph.AddEdge(door, room_b, dictionary={
    "label": "Door connects to Room B",
    "uri": "inst:edge_door_room_b",
    "ontology_class": "top:Relationship",
    "relationship": "connectsTo",
})

TGraph.ValidateOntology(graph)

{'ok': True,
 'errors': [],
 'warnings': [],
 'graph': {'ok': True,
  'errors': [],
  'warnings': [],
  'dictionary': {'label': 'Reasoning Demo Graph',
   'uri': 'inst:demo_graph',
   'ontology_class': 'top:KnowledgeGraph',
   'category': 'graph',
   'ontology_uri': 'http://w3id.org/topologicpy#KnowledgeGraph'}},
 'vertices': [{'ok': True,
   'errors': [],
   'warnings': [],
   'dictionary': {'label': 'Room A',
    'uri': 'inst:room_a',
    'ontology_class': 'top:Room',
    'x': 0.0,
    'y': 0.0,
    'z': 0.0,
    'index': 0,
    'active': True},
   'index': 0},
  {'ok': True,
   'errors': [],
   'warnings': [],
   'dictionary': {'label': 'Room B',
    'uri': 'inst:room_b',
    'ontology_class': 'top:Room',
    'x': 5.0,
    'y': 0.0,
    'z': 0.0,
    'index': 1,
    'active': True},
   'index': 1},
  {'ok': True,
   'errors': [],
   'warnings': [],
   'dictionary': {'label': 'Door 1',
    'uri': 'inst:door_1',
    'ontology_class': 'top:Door',
    'x': 2.5,
    'y': 0.0,
    'z': 0.

## 2. Export to RDF and add ontology axioms

`Reasoner.RDFGraphByTopology` delegates to `Ontology.RDFGraph` when possible, but includes a robust fallback using `TGraph.OntologyTriples`. It also injects the class hierarchy and property domain/range axioms from `Ontology.py`.

In [3]:
rdf_graph = Reasoner.RDFGraphByTopology(
    graph,
    includeGraph=True,
    includeDictionaries=True,
    includeBOT=True,
    includeOntologyAxioms=True,
    silent=True,
)

len(rdf_graph)

858

In [4]:
# Show the asserted types before reasoning.
for subject in ["inst:room_a", "inst:door_1", "inst:demo_graph"]:
    print(subject, "=>", Reasoner.Types(rdf_graph, subject))

inst:room_a => ['bot:Space', 'top:Room']
inst:door_1 => ['bot:Element', 'top:Door']
inst:demo_graph => ['top:KnowledgeGraph']


## 3. Run deterministic RDFS inference

The built-in RDFS profile infers superclass membership, property hierarchy implications, and domain/range types. It does not guess. It only derives facts licensed by explicit triples and ontology axioms.

In [5]:
inferred_graph = Reasoner.Infer(
    rdf_graph,
    profile="rdfs",          # try "owlrl" if owlrl is installed
    includeOntologyAxioms=True,
    includeBOT=True,
)

Reasoner.Summary(rdf_graph, inferred_graph)

{'input_triples': 858, 'output_triples': 948, 'inferred_triples': 90}

In [6]:
# Show inferred types after reasoning.
for subject in ["inst:room_a", "inst:door_1", "inst:demo_graph"]:
    print(subject)
    for t in Reasoner.Types(inferred_graph, subject):
        print("  ", t)

inst:room_a
   bot:Space
   bot:Zone
   top:Cell
   top:Node
   top:Room
   top:Space
   top:Topology
   top:Vertex
   top:Zone
inst:door_1
   bot:Element
   top:Door
   top:Element
   top:Node
   top:Topology
   top:Vertex
inst:demo_graph
   top:Graph
   top:KnowledgeGraph
   top:Topology


## 4. Inspect some newly inferred triples

The difference below contains triples that were not present before reasoning. The exact order may vary because RDF graphs are sets.

In [7]:
for triple in Reasoner.Difference(rdf_graph, inferred_graph, limit=30):
    print(triple)

('inst:door_1', 'rdf:type', 'top:Element')
('inst:door_1', 'rdf:type', 'top:Node')
('top:Stair', 'rdfs:subClassOf', 'bot:Element')
('top:Room', 'rdfs:subClassOf', 'top:Topology')
('top:Space', 'rdfs:subClassOf', 'top:Cell')
('inst:edge_door_room_b', 'rdf:type', 'top:Topology')
('inst:room_a', 'rdf:type', 'bot:Zone')
('top:Node', 'rdfs:subClassOf', 'top:Topology')
('top:Point', 'rdfs:subClassOf', 'top:Topology')
('top:Railing', 'rdfs:subClassOf', 'bot:Element')
('top:VisibilityGraph', 'rdfs:subClassOf', 'top:Graph')
('inst:edge_door_room_b', 'rdf:type', 'top:DirectedRelationship')
('top:DirectedRelationship', 'rdfs:subClassOf', 'top:Topology')
('top:Railing', 'rdfs:subClassOf', 'top:Topology')
('top:Space', 'rdfs:subClassOf', 'top:Topology')
('top:Zone', 'rdfs:subClassOf', 'top:Topology')
('top:Column', 'rdfs:subClassOf', 'top:Topology')
('top:Sensor', 'rdfs:subClassOf', 'bot:Element')
('inst:edge_door_room_b', 'rdf:type', 'top:Edge')
('top:DirectedRelationship', 'rdfs:subClassOf', 'top

## 5. Apply inferred classes back to TGraph dictionaries

The asserted `ontology_class` remains unchanged. Inferred classes are written to `inferred_ontology_classes`, and BOT bridge classes are written to `inferred_bot_classes`.

In [8]:
Reasoner.ApplyInferences(graph, inferred_graph)

for v in TGraph.Vertices(graph, asTopologic=False, activeOnly=True):
    d = v["dictionary"]
    print(d.get("label"))
    print("  asserted:", d.get("ontology_class"))
    print("  inferred top:", d.get("inferred_ontology_classes"))
    print("  inferred bot:", d.get("inferred_bot_classes"))

Room A
  asserted: top:Room
  inferred top: ['top:Cell', 'top:Node', 'top:Space', 'top:Topology', 'top:Vertex', 'top:Zone']
  inferred bot: ['bot:Space', 'bot:Zone']
Room B
  asserted: top:Room
  inferred top: ['top:Cell', 'top:Node', 'top:Space', 'top:Topology', 'top:Vertex', 'top:Zone']
  inferred bot: ['bot:Space', 'bot:Zone']
Door 1
  asserted: top:Door
  inferred top: ['top:Element', 'top:Node', 'top:Topology', 'top:Vertex']
  inferred bot: ['bot:Element']


## 6. Export the inferred graph to Turtle

The inferred RDF graph can be exported and loaded into external RDF/OWL tools.

In [9]:
out_path = Path("topologicpy_reasoning_demo_inferred.ttl")
Reasoner.ExportRDF(inferred_graph, str(out_path), format="turtle", overwrite=True)
out_path

WindowsPath('topologicpy_reasoning_demo_inferred.ttl')

In [10]:
# Print the first part of the Turtle serialization.
turtle = Reasoner.TurtleString(inferred_graph)
print(turtle[:2000])

@prefix bot: <https://w3id.org/bot#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix inst: <http://w3id.org/topologicpy/instance#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix top: <http://w3id.org/topologicpy#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

top:AdjacencyGraph a owl:Class ;
    rdfs:subClassOf top:Graph,
        top:SpatialGraph .

top:Beam a owl:Class ;
    rdfs:subClassOf top:Element,
        top:Topology,
        bot:Element .

top:Building a owl:Class ;
    rdfs:subClassOf top:Cell,
        top:Topology,
        top:Zone,
        bot:Building,
        bot:Zone .

top:CirculationGraph a owl:Class ;
    rdfs:subClassOf top:Graph,
        top:SpatialGraph .

top:CirculationZone a owl:Class ;
    rdfs:subClassOf top:Cell,
        top:Topology,
        top:Zone,
        bot:Zone .

top:Column a owl:Class ;
    rdfs:subClassOf top:El

## Notes for TopologicPy integration

Recommended API placement:

```python
from topologicpy.Reasoner import Reasoner
```

Typical workflow:

```python
rdf_graph = Reasoner.RDFGraphByTopology(tgraph)
inferred_graph = Reasoner.Infer(rdf_graph, profile="rdfs")
tgraph = Reasoner.ApplyInferences(tgraph, inferred_graph)
```

This keeps TopologicPy's geometry/topology operations fast while adding a deterministic, auditable semantic reasoning layer when needed.